# Stage 2 — GRPO RL 후학습 실습

NTP로 사전학습된 AutoGaze를 **GRPO(Group Relative Policy Optimization)** 강화학습으로  
더 나은 가이즈 전략을 스스로 탐색하도록 학습합니다.

## GRPO 알고리즘 개요

```text
입력 비디오 v
    │
    ├── 정책 π(θ) 으로 G개의 가이즈 시퀀스 샘플링
    │     a₁, a₂, ..., aG  (group_size = G)
    │
    ├── 각 시퀀스의 보상 계산
    │     r_i = -L_recon(VideoMAE, v, a_i)  ← 재건 손실이 낮을수록 높은 보상
    │
    ├── 그룹 내 상대적 이득 (Advantage)
    │     A_i = r_i - mean(r₁..rG)
    │
    └── 정책 경사 업데이트
          ∇θ ∝ Σ A_i · ∇θ log π(a_i | v)
```

**NTP vs RL 핵심 차이**

| 항목 | Stage 1 NTP | Stage 2 RL |
| --- | --- | --- |
| 교사 신호 | GT 레이블 (`gazing_labels.json`) | VideoMAE 재건 보상 |
| 가이즈 비율 | 0.1 (낮게 시작) | 0.75 (더 많이 탐색) |
| 배치당 샘플 | 1개 | group_size (4~12)개 |
| 에폭 수 | 150 | 1 (계산량이 G배) |
| 목적 | 기본 능력 습득 | 재건 품질 최적화 |

**다루는 내용**
1. 사전 준비 및 NTP 체크포인트 확인
2. GRPO 알고리즘 파라미터 이해
3. RL 학습 실행
4. 학습 중 보상 변화 모니터링
5. NTP vs RL 모델 비교
6. 파라미터 실험 (group_size, discount_factor)

**사전 조건**
```bash
# 02_train_ntp_ko.ipynb 또는 scripts/train_ntp_single_gpu.sh 를 먼저 실행해
# exps/ntp_tutorial/checkpoint_latest_gaze 가 생성되어 있어야 합니다.
```

---
## 0. 환경 확인 및 경로 설정

In [ ]:
import platform, sys
import matplotlib
import matplotlib.font_manager as fm
import torch
import numpy as np

# 한글 폰트 설정
def _setup_korean_font():
    _sys = platform.system()
    if _sys == 'Darwin':
        matplotlib.rcParams['font.family'] = 'AppleGothic'
        _font = 'AppleGothic'
    elif _sys == 'Windows':
        matplotlib.rcParams['font.family'] = 'Malgun Gothic'
        _font = 'Malgun Gothic'
    else:
        _candidates = [f.name for f in fm.fontManager.ttflist
                       if any(k in f.name for k in ('Nanum', 'Gothic', 'Batang'))]
        if _candidates:
            matplotlib.rcParams['font.family'] = _candidates[0]
            _font = _candidates[0]
        else:
            print("⚠  한글 폰트 없음 — sudo apt-get install fonts-nanum")
            return
    matplotlib.rcParams['axes.unicode_minus'] = False
    print(f"[폰트] {_font}")

_setup_korean_font()
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Device : {'CUDA' if torch.cuda.is_available() else 'MPS' if torch.backends.mps.is_available() else 'CPU'}")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

# ── 경로 설정 ────────────────────────────────────────────────────
ROOT          = Path("..")
DATA_ROOT     = ROOT / "data/AutoGaze-Training-Data"
VIDEOMAE_PT   = ROOT / "weights/VideoMAE_AutoGaze/videomae.pt"

# Stage 1 NTP 체크포인트 (Stage 2 초기 가중치)
NTP_EXP_NAME  = "ntp_tutorial"       # 02_train_ntp_ko.ipynb 에서 사용한 이름
NTP_CKPT_DIR  = ROOT / f"exps/{NTP_EXP_NAME}/checkpoint_latest_gaze"

# Stage 2 RL 실험 이름
RL_EXP_NAME   = "rl_tutorial"
RL_EXP_DIR    = ROOT / f"exps/{RL_EXP_NAME}"

DATASET_PATHS = [p for p in [
    DATA_ROOT / "InternVid_res448_250K",
] if p.exists()]
# ─────────────────────────────────────────────────────────────────

print("경로 확인")
checks = {
    "VideoMAE 가중치 ": VIDEOMAE_PT,
    "NTP 체크포인트  ": NTP_CKPT_DIR,
}
for label, path in checks.items():
    status = "✓" if path.exists() else "✗ (없음)"
    print(f"  {label}: {status}")

if not NTP_CKPT_DIR.exists():
    print()
    print("⚠  NTP 체크포인트가 없습니다. 먼저 실행하세요:")
    print("   notebooks/02_train_ntp_ko.ipynb  또는")
    print("   bash scripts/train_ntp_single_gpu.sh <데이터> weights/VideoMAE_AutoGaze/videomae.pt")

---
## 1. GRPO 파라미터 이해

GRPO의 핵심 파라미터가 학습에 어떤 영향을 미치는지 이해합니다.

In [ ]:
# group_size 영향 시뮬레이션
# group_size가 클수록 advantage 추정이 안정적이지만 메모리/시간이 증가

np.random.seed(42)

def simulate_grpo_advantage(group_size, n_trials=1000, reward_std=0.3):
    """그룹 상대 이득의 분산을 시뮬레이션."""
    advantages = []
    for _ in range(n_trials):
        rewards = np.random.normal(0, reward_std, size=group_size)
        adv = rewards - rewards.mean()
        advantages.extend(adv.tolist())
    return np.array(advantages)

group_sizes = [2, 4, 8, 12, 16]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(group_sizes)))
stds = []

for gs, color in zip(group_sizes, colors):
    advs = simulate_grpo_advantage(gs)
    axes[0].hist(advs, bins=50, alpha=0.5, label=f'G={gs}', color=color, density=True)
    stds.append(advs.std())

axes[0].set_xlabel('Advantage 값')
axes[0].set_ylabel('밀도')
axes[0].set_title('group_size별 Advantage 분포')
axes[0].legend()

axes[1].plot(group_sizes, stds, marker='o', color='steelblue', linewidth=2)
axes[1].set_xlabel('group_size (G)')
axes[1].set_ylabel('Advantage 표준편차')
axes[1].set_title('group_size ↑ → Advantage 분산 ↓ (더 안정적)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("group_size가 클수록 advantage가 안정적이지만 GPU 메모리가 G배 필요합니다.")
print("  단일 GPU: group_size=4 권장")
print("  8+ GPU  : group_size=12 (논문 설정)")

In [ ]:
# discount_factor 영향 시뮬레이션
# discount_factor가 1에 가까울수록 보상이 궤적 전체에 고르게 기여

T = 20  # 가이즈 스텝 수
terminal_reward = 1.0  # 마지막 스텝에서 받은 보상

fig, ax = plt.subplots(figsize=(10, 4))

gammas = [0.9, 0.95, 0.99, 0.995, 1.0]
colors2 = plt.cm.plasma(np.linspace(0.1, 0.9, len(gammas)))

for gamma, color in zip(gammas, colors2):
    # 각 스텝에서의 할인된 보상
    discounted = [terminal_reward * (gamma ** (T - 1 - t)) for t in range(T)]
    ax.plot(range(T), discounted, label=f'γ={gamma}', color=color, linewidth=2)

ax.set_xlabel('가이즈 스텝 t')
ax.set_ylabel('할인된 보상')
ax.set_title('discount_factor(γ)별 보상 전파: 마지막 스텝의 보상이 이전 스텝에 얼마나 기여하는가')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("γ=1.0 : 모든 스텝에 동일한 보상 → 각 가이즈 결정이 동등하게 중요")
print("γ=0.9 : 초기 스텝은 거의 보상 없음 → 마지막 결정만 중요")
print("AutoGaze 논문 설정: γ=0.995 (거의 균등, 약간 최신 스텝 강조)")

In [ ]:
# RL 학습 핵심 파라미터 요약
rl_params = {
    "algorithm": {
        "group_size"                      : ("4 (단일GPU) / 12 (논문)", "입력당 샘플 시퀀스 수"),
        "discount_factor"                 : (0.995, "시간 할인 계수"),
        "optimize_task_loss_prediction"   : (True, "재건 손실 예측 학습"),
    },
    "model": {
        "gazing_ratio_config.fixed"       : (0.75, "NTP보다 높은 비율 탐색"),
        "gazing_ratio_each_frame.self"    : ("on-policy", "자체 결정한 프레임별 예산"),
        "task_loss_requirement.uniform"   : ("0.5~1.0", "다양한 품질 요건 학습"),
    },
    "trainer": {
        "lr"                              : (5e-4, "NTP와 동일"),
        "n_epochs"                        : (1, "RL은 1 에폭 (G배 계산)"),
        "batch_size"                      : ("8 (단일) / 64 (논문)", ""),
        "train_task"                      : (False, "VideoMAE 동결 유지"),
        "gaze_weights"                    : ("NTP 체크포인트", "Stage 1 결과 로드"),
    },
}

for group, params in rl_params.items():
    print(f"\n[{group}]")
    for key, (val, desc) in params.items():
        print(f"  {key:<45s} = {str(val):<25s} # {desc}")

---
## 2. RL 학습 실행

### 2-A. 소규모 테스트

In [ ]:
import subprocess, shlex

def run_training(cmd, cwd=None, timeout=600):
    cwd = cwd or str(ROOT)
    print(f"$ {cmd[:120]}...\n" if len(cmd) > 120 else f"$ {cmd}\n")
    print("─" * 60)
    try:
        proc = subprocess.Popen(
            shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, cwd=cwd
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=timeout)
        print("─" * 60)
        print(f"종료 코드: {proc.returncode}")
        return proc.returncode
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"\n[타임아웃 {timeout}s]")
        return -1

valid_paths = [str(p) for p in DATASET_PATHS if p.exists()]
DATA_ROOTS_STR = ",".join(valid_paths) if valid_paths else "<데이터셋 경로>"

print(f"데이터: {DATA_ROOTS_STR}")
print(f"NTP 체크포인트: {NTP_CKPT_DIR}  ({'✓' if NTP_CKPT_DIR.exists() else '✗ 없음'})")

In [ ]:
if valid_paths and VIDEOMAE_PT.exists() and NTP_CKPT_DIR.exists():
    cmd = f"""
python -m autogaze.train
  --config-name video_folder_video_mae_reconstruction_ar_gaze_grpo
  dataset.root='{DATA_ROOTS_STR}'
  dataset.clip_len=16
  model.gazing_ratio_config.sample_strategy_during_training=fixed
  model.gazing_ratio_config.sample_strategy_during_inference=fixed
  model.gazing_ratio_config.fixed.gazing_ratio=0.75
  model.gazing_ratio_each_frame_config.sample_strategy_during_training=self
  model.scales=32+64+112+224
  model.num_vision_tokens_each_frame=265
  model.has_task_loss_requirement_during_training=False
  model.has_task_loss_requirement_during_inference=True
  model.task_loss_requirement_config.sample_strategy_during_training=uniform
  model.task_loss_requirement_config.sample_strategy_during_inference=fixed
  model.task_loss_requirement_config.fixed.task_loss_requirement=0.7
  model.task_loss_requirement_config.uniform.task_loss_requirement_min=0.5
  model.task_loss_requirement_config.uniform.task_loss_requirement_max=1.0
  model.gaze_model_config.gaze_decoder_config.num_multi_token_pred=10
  task.recon_model=facebook/vit-mae-large
  task.recon_sample_rate=0.125
  task.recon_model_config.loss_type=l1+dinov2_reg+siglip2
  task.recon_model_config.loss_weights=1+0.3+0.3
  task.scales=32+64+112+224
  algorithm.group_size=2
  algorithm.discount_factor=0.995
  algorithm.optimize_task_loss_prediction=True
  trainer.train_gaze=True
  trainer.train_task=False
  trainer.detach_task=True
  trainer.lr=5e-4
  trainer.n_epochs=1
  trainer.batch_size=4
  trainer.per_gpu_max_batch_size=1
  trainer.val_nsteps=10
  trainer.save_nsteps=10
  trainer.temp_schedule_args.exp.temp_start=3
  trainer.temp_schedule_args.exp.temp_end=0.3
  trainer.task_weights={VIDEOMAE_PT}
  trainer.gaze_weights={NTP_CKPT_DIR}
  trainer.exp_name={RL_EXP_NAME}
""".replace('\n', ' ').replace('  ', ' ').strip()

    run_training(cmd, timeout=300)
else:
    missing = []
    if not valid_paths:          missing.append("데이터셋")
    if not VIDEOMAE_PT.exists(): missing.append("VideoMAE 가중치")
    if not NTP_CKPT_DIR.exists():missing.append("NTP 체크포인트")
    print(f"⚠  필요한 파일 없음: {', '.join(missing)}")

### 2-B. 실제 학습 스크립트 생성

In [ ]:
script_content = f"""#!/usr/bin/env bash
# 자동 생성: Stage 2 GRPO RL 학습
set -euo pipefail

bash scripts/train_rl_single_gpu.sh \\
    "{DATA_ROOTS_STR}" \\
    "{VIDEOMAE_PT}" \\
    "{NTP_CKPT_DIR}"
"""

script_path = ROOT / 'scripts/my_train_rl.sh'
script_path.write_text(script_content)
script_path.chmod(0o755)
print(f"RL 학습 스크립트 생성: {script_path}")
print()
print("실행 방법 (터미널에서):")
print(f"  bash {script_path.relative_to(ROOT)}")
print()
print("백그라운드 실행:")
print(f"  nohup bash {script_path.relative_to(ROOT)} > rl_train.log 2>&1 &")
print(f"  tail -f rl_train.log")

---
## 3. 학습 모니터링 — 보상 변화 추적

In [ ]:
import re

def parse_rl_log(log_path):
    """학습 로그에서 보상/손실 지표 파싱."""
    steps, rewards, losses, patch_counts = [], [], [], []
    pattern_step    = re.compile(r'step[\s=:]*(\d+)', re.I)
    pattern_reward  = re.compile(r'reward[\s=:]*([\-\d.]+)', re.I)
    pattern_loss    = re.compile(r'(?:policy_)?loss[\s=:]*([\-\d.]+)', re.I)
    pattern_patches = re.compile(r'gazed[\s=:]*(\d+)', re.I)

    with open(log_path) as f:
        for line in f:
            m_step = pattern_step.search(line)
            m_rew  = pattern_reward.search(line)
            m_loss = pattern_loss.search(line)
            m_pat  = pattern_patches.search(line)

            if m_step and m_rew:
                steps.append(int(m_step.group(1)))
                rewards.append(float(m_rew.group(1)))
                if m_loss:  losses.append(float(m_loss.group(1)))
                if m_pat:   patch_counts.append(int(m_pat.group(1)))

    return steps, rewards, losses, patch_counts

# 로그 파일 탐색
log_files = list(RL_EXP_DIR.rglob('*.log')) if RL_EXP_DIR.exists() else []

if log_files:
    steps, rewards, losses, patches = parse_rl_log(log_files[0])
    if steps:
        fig, axes = plt.subplots(1, 2 + (1 if patches else 0), figsize=(14, 4))

        axes[0].plot(steps, rewards, color='steelblue', linewidth=1.5)
        axes[0].set_xlabel('학습 스텝')
        axes[0].set_ylabel('평균 보상 (-재건손실)')
        axes[0].set_title('RL 학습 보상 변화')
        axes[0].grid(alpha=0.3)

        if losses:
            axes[1].plot(steps[:len(losses)], losses, color='tomato', linewidth=1.5)
            axes[1].set_xlabel('학습 스텝')
            axes[1].set_ylabel('정책 손실')
            axes[1].set_title('정책 손실 변화')
            axes[1].grid(alpha=0.3)

        if patches:
            axes[-1].plot(steps[:len(patches)], patches, color='seagreen', linewidth=1.5)
            axes[-1].set_xlabel('학습 스텝')
            axes[-1].set_ylabel('평균 선택 패치 수')
            axes[-1].set_title('선택 패치 수 변화 (효율 탐색)')
            axes[-1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()
    else:
        print("로그에서 지표를 파싱하지 못했습니다.")
else:
    print(f"로그 파일 없음: {RL_EXP_DIR}")
    print("학습을 먼저 실행하거나, 아래 시뮬레이션으로 예상 곡선을 확인하세요.")

In [ ]:
# RL 학습 예상 곡선 시뮬레이션 (실제 로그가 없을 때)
np.random.seed(0)
T_sim = 500
t_arr = np.arange(T_sim)

# 보상: 초기 탐색 후 점진적 향상
reward_sim = -0.8 + 0.4 * (1 - np.exp(-t_arr / 100)) + np.random.normal(0, 0.05, T_sim)
reward_sim = np.convolve(reward_sim, np.ones(20)/20, mode='same')

# 선택 패치 수: 처음엔 많이 보다가 점차 효율적으로
patches_sim = 180 - 60 * (1 - np.exp(-t_arr / 150)) + np.random.normal(0, 5, T_sim)
patches_sim = np.convolve(patches_sim, np.ones(20)/20, mode='same')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_arr, reward_sim, color='steelblue', linewidth=2)
axes[0].axhline(y=-0.8, color='gray', linestyle='--', alpha=0.5, label='NTP 초기 보상')
axes[0].axhline(y=-0.4, color='tomato', linestyle='--', alpha=0.5, label='RL 수렴 후')
axes[0].set_xlabel('학습 스텝')
axes[0].set_ylabel('평균 보상')
axes[0].set_title('RL 보상 변화 (시뮬레이션)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(t_arr, patches_sim, color='seagreen', linewidth=2)
axes[1].axhline(y=180, color='gray', linestyle='--', alpha=0.5, label='초기')
axes[1].axhline(y=120, color='tomato', linestyle='--', alpha=0.5, label='수렴 후 (효율 증가)')
axes[1].set_xlabel('학습 스텝')
axes[1].set_ylabel('평균 선택 패치 수')
axes[1].set_title('선택 패치 수 변화 (시뮬레이션)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('GRPO RL 학습 예상 곡선 (실제 결과는 다를 수 있음)', fontsize=11)
plt.tight_layout()
plt.show()

---
## 4. NTP vs RL 모델 비교

동일 비디오에서 NTP 학습 모델과 RL 학습 모델의 가이즈 패턴을 비교합니다.

In [ ]:
import av
import sys; sys.path.insert(0, "..")
from autogaze.models.autogaze import AutoGaze, AutoGazeImageProcessor
from autogaze.utils import get_device, UnNormalize
from autogaze.datasets.video_utils import (
    read_video_pyav, sample_frame_indices, process_video_frames,
    transform_video_for_pytorch,
)
import torch.nn.functional as F
import matplotlib.patches as mpatches

device = get_device()

# NTP 모델 로드
NTP_MODEL_PATH = str(NTP_CKPT_DIR) if NTP_CKPT_DIR.exists() else "nvidia/AutoGaze"
print(f"NTP 모델: {NTP_MODEL_PATH}")
ntp_transform = AutoGazeImageProcessor.from_pretrained(NTP_MODEL_PATH)
ntp_model     = AutoGaze.from_pretrained(NTP_MODEL_PATH).to(device).eval()

# RL 모델 로드 (있으면)
RL_CKPT_DIR  = RL_EXP_DIR / 'checkpoint_latest_gaze'
RL_MODEL_PATH = str(RL_CKPT_DIR) if RL_CKPT_DIR.exists() else None

if RL_MODEL_PATH:
    print(f"RL 모델: {RL_MODEL_PATH}")
    rl_transform = AutoGazeImageProcessor.from_pretrained(RL_MODEL_PATH)
    rl_model     = AutoGaze.from_pretrained(RL_MODEL_PATH).to(device).eval()
    RL_AVAILABLE = True
else:
    print("RL 체크포인트 없음 → NTP 모델과 다른 파라미터 설정으로 비교")
    rl_transform = ntp_transform
    rl_model     = ntp_model
    RL_AVAILABLE = False

print("모델 로드 완료 ✓")

In [ ]:
# 예제 비디오 로드
example_video = ROOT / 'assets/example_input.mp4'

if not example_video.exists():
    print(f"예제 비디오 없음: {example_video}")
else:
    container = av.open(str(example_video))
    total = container.streams.video[0].frames
    indices = sample_frame_indices(16, 1, total, False)
    raw = read_video_pyav(container, indices)
    container.close()
    raw = process_video_frames(raw, 16)

    T = len(raw)

    # NTP 인퍼런스
    v_ntp = transform_video_for_pytorch(raw, ntp_transform)[None].to(device)
    with torch.inference_mode():
        out_ntp = ntp_model({"video": v_ntp}, gazing_ratio=0.75, task_loss_requirement=0.7)

    # RL 인퍼런스
    v_rl = transform_video_for_pytorch(raw, rl_transform)[None].to(device)
    with torch.inference_mode():
        if RL_AVAILABLE:
            out_rl = rl_model({"video": v_rl}, gazing_ratio=0.75, task_loss_requirement=0.7)
        else:
            # RL 없으면 gazing_ratio를 낮춰서 비교 (파라미터 차이 시연)
            out_rl = ntp_model({"video": v_rl}, gazing_ratio=0.3, task_loss_requirement=0.7)

    n_ntp = int((~out_ntp['if_padded_gazing']).sum())
    n_rl  = int((~out_rl['if_padded_gazing']).sum())
    n_tot = ntp_model.config.num_vision_tokens_each_frame * T

    label_ntp = "NTP 모델"
    label_rl  = "RL 모델" if RL_AVAILABLE else "NTP (ratio=0.3)"

    print(f"{label_ntp}: {n_ntp} / {n_tot} 패치 ({100*n_ntp/n_tot:.1f}%)")
    print(f"{label_rl }: {n_rl}  / {n_tot} 패치 ({100*n_rl/n_tot:.1f}%)")

In [ ]:
if example_video.exists():
    SCALES = [int(s) for s in ntp_model.config.scales.split("+")]
    largest_si = len(SCALES) - 1
    largest_scale = SCALES[largest_si]

    unnorm = UnNormalize(
        ntp_transform.image_mean, ntp_transform.image_std,
        getattr(ntp_transform, 'rescale_factor', 1/255.0)
    )
    video_np = unnorm(transform_video_for_pytorch(raw, ntp_transform)).cpu().float().numpy()

    def make_overlay(frame_chw, mask_hw, scale):
        ft = torch.from_numpy(frame_chw).unsqueeze(0)
        fs = F.interpolate(ft, size=(scale, scale), mode='bicubic', align_corners=False)
        fs = fs.squeeze().clamp(0, 1).numpy()
        m = F.interpolate(
            torch.from_numpy(mask_hw).unsqueeze(0).unsqueeze(0).float(),
            size=(scale, scale), mode='nearest'
        ).squeeze().numpy()
        display = fs * (0.25 + 0.75 * m[None])
        return (np.clip(display.transpose(1, 2, 0), 0, 1) * 255).astype(np.uint8)

    # 처음 8프레임 비교
    T_show = min(T, 8)
    fig, axes = plt.subplots(3, T_show, figsize=(T_show * 2.5, 7))

    mask_ntp = out_ntp['gazing_mask'][largest_si][0]  # (T, N)
    mask_rl  = out_rl['gazing_mask'][largest_si][0]
    pg = int(mask_ntp.shape[-1] ** 0.5)

    num_each_ntp = out_ntp['num_gazing_each_frame']
    if_pad_ntp   = out_ntp['if_padded_gazing']
    num_each_rl  = out_rl['num_gazing_each_frame']
    if_pad_rl    = out_rl['if_padded_gazing']

    # 실제 패치 수 계산
    def real_per_frame(num_each, if_pad):
        counts = []
        offset = 0
        for cnt in num_each.tolist():
            pad = if_pad[0][offset: offset + cnt]
            counts.append(int((~pad).sum()))
            offset += cnt
        return counts

    ntp_counts = real_per_frame(num_each_ntp, if_pad_ntp)
    rl_counts  = real_per_frame(num_each_rl,  if_pad_rl)

    for t in range(T_show):
        # 원본
        axes[0, t].imshow(raw[t])
        axes[0, t].set_title(f'F{t+1}\n원본', fontsize=7)
        axes[0, t].axis('off')

        # NTP 가이즈
        m_ntp = mask_ntp[t].reshape(pg, pg).cpu().float().numpy()
        axes[1, t].imshow(make_overlay(video_np[t], m_ntp, largest_scale))
        axes[1, t].set_title(f'{label_ntp}\n({ntp_counts[t]}p)', fontsize=7)
        axes[1, t].axis('off')

        # RL 가이즈
        m_rl = mask_rl[t].reshape(pg, pg).cpu().float().numpy()
        axes[2, t].imshow(make_overlay(video_np[t], m_rl, largest_scale))
        axes[2, t].set_title(f'{label_rl}\n({rl_counts[t]}p)', fontsize=7)
        axes[2, t].axis('off')

    plt.suptitle(f'Scale-{largest_scale} 가이즈 비교: {label_ntp} vs {label_rl}', fontsize=12)
    plt.tight_layout()
    plt.show()

    # 프레임별 패치 수 비교
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(T)
    w = 0.35
    ax.bar(x - w/2, ntp_counts, w, label=label_ntp, color='steelblue', edgecolor='white')
    ax.bar(x + w/2, rl_counts,  w, label=label_rl,  color='tomato',    edgecolor='white')
    ax.set_xlabel('프레임 번호')
    ax.set_ylabel('선택 패치 수')
    ax.set_title(f'프레임별 선택 패치 수 비교 (NTP 총={sum(ntp_counts)}, RL 총={sum(rl_counts)})')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 5. group_size 파라미터 실험

`group_size`가 가이즈 다양성에 미치는 영향을 관찰합니다.

In [ ]:
# 동일 입력에서 여러 번 샘플링해 가이즈 다양성 측정
if example_video.exists():
    N_SAMPLES = 6
    SCALES = [int(s) for s in ntp_model.config.scales.split("+")]

    samples = []
    for i in range(N_SAMPLES):
        with torch.inference_mode():
            # temperature=1 (기본값)으로 다양한 샘플 생성
            out_s = ntp_model(
                {"video": v_ntp},
                gazing_ratio=0.75,
                task_loss_requirement=None,
            )
        samples.append(out_s)

    # 샘플간 가이즈 패턴 일치율 (자카드 유사도)
    def jaccard(set1, set2):
        inter = len(set1 & set2)
        union = len(set1 | set2)
        return inter / union if union > 0 else 0

    def get_gazed_set(out, frame=0):
        m = out['gazing_mask'][-1][0][frame]  # scale_224, frame 0
        pg = int(m.shape[-1] ** 0.5)
        mask = m.reshape(pg, pg).cpu().float().numpy() > 0.5
        return set(zip(*np.where(mask)))

    # 프레임 0의 가이즈 패턴 유사도 행렬
    sim_matrix = np.zeros((N_SAMPLES, N_SAMPLES))
    for i in range(N_SAMPLES):
        for j in range(N_SAMPLES):
            sim_matrix[i, j] = jaccard(get_gazed_set(samples[i]), get_gazed_set(samples[j]))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    im = axes[0].imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[0])
    axes[0].set_xticks(range(N_SAMPLES))
    axes[0].set_yticks(range(N_SAMPLES))
    axes[0].set_xticklabels([f'샘플{i+1}' for i in range(N_SAMPLES)])
    axes[0].set_yticklabels([f'샘플{i+1}' for i in range(N_SAMPLES)])
    for i in range(N_SAMPLES):
        for j in range(N_SAMPLES):
            axes[0].text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
    axes[0].set_title(f'가이즈 패턴 자카드 유사도\n(프레임 1, {N_SAMPLES}회 샘플링)')

    # 샘플별 Scale-224 가이즈 시각화
    T_show2 = min(N_SAMPLES, 6)
    fig2, ax2_arr = plt.subplots(1, T_show2, figsize=(T_show2 * 2.5, 2.5))
    pg = int(samples[0]['gazing_mask'][-1][0][0].shape[-1] ** 0.5)

    for si in range(T_show2):
        m = samples[si]['gazing_mask'][-1][0][0].reshape(pg, pg).cpu().float().numpy()
        n = int(m.sum())
        img = make_overlay(video_np[0], m, largest_scale)
        ax2_arr[si].imshow(img)
        ax2_arr[si].set_title(f'샘플 {si+1}\n({n}패치)', fontsize=8)
        ax2_arr[si].axis('off')

    plt.suptitle('동일 입력에서의 가이즈 다양성 (GRPO가 이 다양성을 활용)', fontsize=10)
    plt.tight_layout()
    plt.show()

    # 유사도 통계
    off_diag = sim_matrix[np.triu_indices(N_SAMPLES, k=1)]
    print(f"\n샘플간 평균 자카드 유사도: {off_diag.mean():.3f} ± {off_diag.std():.3f}")
    print(f"→ GRPO는 이 {N_SAMPLES}개 샘플의 재건 품질을 비교해 더 나은 방향으로 학습")

---
## 6. 체크포인트 확인 및 다음 단계

In [ ]:
# RL 학습 후 체크포인트 확인
if RL_EXP_DIR.exists():
    print(f"=== {RL_EXP_DIR} ===")
    for ckpt in sorted(RL_EXP_DIR.iterdir()):
        if ckpt.is_dir():
            files = list(ckpt.iterdir())
            size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
            print(f"  {ckpt.name}/  ({size_mb:.1f} MB)")
else:
    print(f"RL 실험 디렉터리 없음: {RL_EXP_DIR}")

print("\n─" * 30)
print("RL 학습 완료 후 인퍼런스 사용법:")
print()
rl_ckpt_path = RL_EXP_DIR / 'checkpoint_latest_gaze'
print(f"  python -m autogaze.infer assets/example_input.mp4 \\")
print(f"      --model-path {rl_ckpt_path} \\")
print(f"      --output-format frames,video --output-dir results/rl_test")

---
## 정리

| 항목 | NTP (Stage 1) | RL/GRPO (Stage 2) |
| --- | --- | --- |
| 교사 신호 | GT 레이블 | VideoMAE 재건 보상 |
| 핵심 설정 | `video_folder_..._ar_gaze_ntp` | `video_folder_..._ar_gaze_grpo` |
| 가이즈 비율 | 0.1 | 0.75 |
| group_size | 해당 없음 | 4 (단일 GPU) / 12 (논문) |
| discount_factor | 해당 없음 | 0.995 |
| 에폭 | 150 | 1 |
| 체크포인트 | `exps/ntp_*/checkpoint_latest_gaze` | `exps/rl_*/checkpoint_latest_gaze` |

**전체 학습 파이프라인 요약:**
```text
데이터 다운로드 (download_data.sh)
    → Stage 1: NTP 사전학습 (train_ntp_single_gpu.sh)  [~수 시간]
    → Stage 2: GRPO RL 후학습 (train_rl_single_gpu.sh) [~수 시간]
    → 학습된 모델로 인퍼런스 (infer.py --model-path exps/rl_*/checkpoint_latest_gaze)
```

**참고 자료**
- `GUIDE_KO.md` — 전체 한글 사용 가이드
- `TRAIN.md` — 영문 학습 문서
- `01_autogaze_tutorial_ko.ipynb` — 인퍼런스 기본 실습